# AlphaTrain Self-Play (Colab GPU)

Generate self-play training data using Neural MCTS on Colab GPU.
Each game produces (observation, MCTS visit policy, TD value target) per move.

**Setup:**
1. Upload `colorlines_selfplay.tar.gz` to Google Drive root
2. Upload `alphatrain_td_best.pt` (model) to Google Drive root
3. Run all cells

**Build tarball locally:**
```bash
tar czf colorlines_selfplay.tar.gz \
    --exclude='__pycache__' --exclude='*.nbc' --exclude='*.nbi' \
    --exclude='data' --exclude='.venv' --exclude='.git' \
    --exclude='*.tar.gz' --exclude='selfplay_data' \
    -C /Users/andreis/local/source/colorlines98 \
    alphatrain game
```

**Resume-safe:** Games save individually to Drive. If Colab disconnects, just re-run — completed games are skipped automatically.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil

# Extract code
!cp /content/drive/MyDrive/colorlines_selfplay.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay.tar.gz

# Copy model
os.makedirs('/content/alphatrain/data', exist_ok=True)
shutil.copy('/content/drive/MyDrive/alphatrain_td_best.pt',
            '/content/alphatrain/data/alphatrain_td_best.pt')

# Install deps
!pip install -q numpy numba scipy pytest

print('\n=== Setup complete ===')
print(f'Model: {os.path.getsize("/content/alphatrain/data/alphatrain_td_best.pt") / 1e6:.1f} MB')

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {mem:.1f} GB')

# Sanity tests
!cd /content && python -m pytest alphatrain/tests/test_mcts.py -v --tb=short 2>&1 | tail -10

In [ ]:
# Quick benchmark: measure per-turn speed on this GPU
import sys
sys.path.insert(0, '/content')

import time, numpy as np, torch
from game.board import ColorLinesGame
from alphatrain.mcts import MCTS
from alphatrain.evaluate import load_model

dev = torch.device('cuda')
net, ms = load_model('/content/alphatrain/data/alphatrain_td_best.pt',
                     dev, fp16=True, jit_trace=True)
mcts = MCTS(net, dev, max_score=ms, num_simulations=800,
            batch_size=8, top_k=30, c_puct=2.5)

# Warmup + benchmark 20 turns
game = ColorLinesGame(seed=42)
game.reset()
times = []
for i in range(20):
    if game.game_over:
        break
    t0 = time.time()
    action = mcts.search(game)
    times.append(time.time() - t0)
    if action is None:
        break
    game.move(action[0], action[1])

ms_per_turn = np.mean(times[2:]) * 1000  # skip warmup
est_game_min = ms_per_turn * 500 / 1000 / 60  # ~500 turns avg
est_500_hr = est_game_min * 500 / 60
print(f'\n=== GPU Benchmark ===')
print(f'{ms_per_turn:.0f} ms/turn (800 sims, bs=8)')
print(f'Est. {est_game_min:.1f} min/game, {est_500_hr:.1f}h for 500 games')

del net, mcts  # free GPU memory

In [ ]:
#@title Self-Play Config { run: "auto" }
SEED_START = 1000  #@param {type:"integer"}
SEED_END = 1500    #@param {type:"integer"}
SIMS = 800         #@param {type:"integer"}
BATCH_SIZE = 8     #@param {type:"integer"}

SAVE_DIR = '/content/drive/MyDrive/alphatrain/games'
MODEL = '/content/alphatrain/data/alphatrain_td_best.pt'

n_games = SEED_END - SEED_START
print(f'Will generate {n_games} games (seeds {SEED_START}-{SEED_END-1})')
print(f'Saving to: {SAVE_DIR}/')

# Check existing
import os
os.makedirs(SAVE_DIR, exist_ok=True)
existing = [f for f in os.listdir(SAVE_DIR) if f.startswith('game_') and f.endswith('.pt')]
if existing:
    print(f'{len(existing)} games already in save dir (will be skipped)')

In [ ]:
# Run self-play (resume-safe: skips completed seeds)
# GPU server mode: 4 CPU workers share 1 GPU for ~3x throughput
%cd /content
!python -m alphatrain.scripts.selfplay \
    --model {MODEL} \
    --seed-start {SEED_START} --seed-end {SEED_END} \
    --sims {SIMS} --batch-size {BATCH_SIZE} \
    --device cuda --workers 4 \
    --save-dir {SAVE_DIR}

In [ ]:
# Summary of completed games
import torch, numpy as np, os

SAVE_DIR = '/content/drive/MyDrive/alphatrain/games'
files = sorted([f for f in os.listdir(SAVE_DIR)
                if f.startswith('game_') and f.endswith('.pt')])

scores, turns, states = [], [], 0
for f in files:
    g = torch.load(os.path.join(SAVE_DIR, f), weights_only=False)
    scores.append(g['score'])
    turns.append(g['turns'])
    states += g['turns']

scores = np.array(scores)
print(f'=== Self-Play Summary ===')
print(f'Games: {len(files)}')
print(f'States: {states:,}')
print(f'Score: mean={scores.mean():.0f}, median={np.median(scores):.0f}, '
      f'min={scores.min()}, max={scores.max()}')
print(f'Turns: mean={np.mean(turns):.0f}')
print(f'\nSample files: {files[:3]} ... {files[-3:]}')